# FMML Module 1 Lab 2: Logistic Regression on Water Quality Dataset

**Foundations of Modern Machine Learning (FMML)**

This notebook applies **Logistic Regression** to the Water Quality dataset to predict whether water is safe (`is_safe = 1`) or not (`is_safe = 0`).

### Strict Pipeline (must follow exactly to reproduce coefficients)
1. Load dataset — no shuffling or row reordering
2. Convert all columns to numeric (`pd.to_numeric(errors='coerce')`)
3. Drop rows where `is_safe` is NaN; cast to `int`
4. Select 20 features in the exact prescribed order
5. Impute missing feature values with column mean (`SimpleImputer(strategy='mean')`)
6. Scale features with `StandardScaler` (Z-score normalisation)
7. Fit `LogisticRegression(solver='liblinear', penalty='l2', C=1, random_state=42, max_iter=1000, tol=1e-8)` on the **full** dataset
8. Report decision-boundary equation and classification rule

---

## Step 0 — Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

print("Imports OK")

## Step 1 — Data Loading

In [ ]:
# Load dataset — preserve original row order (no shuffle, no sort)
df = pd.read_csv('waterQuality1.csv')

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

## Step 2 — Data Cleaning: convert all columns to numeric

In [ ]:
# Convert every column to numeric; invalid strings (e.g. #NUM!) become NaN
df = df.apply(pd.to_numeric, errors='coerce')

print("NaN counts per column after conversion:")
print(df.isna().sum())
df.dtypes

## Step 3 — Target Variable Handling

In [ ]:
# Drop rows where target is missing, then cast to int
df = df.dropna(subset=['is_safe'])
df['is_safe'] = df['is_safe'].astype(int)

print(f"Rows after dropping missing targets: {len(df)}")
print(f"Class distribution:\n{df['is_safe'].value_counts()}")

## Step 4 — Feature Selection (strict order)

In [ ]:
# DO NOT change this order — it affects the coefficient vector
FEATURE_COLS = [
    'aluminium', 'ammonia', 'arsenic', 'barium', 'cadmium',
    'chloramine', 'chromium', 'copper', 'flouride', 'bacteria',
    'viruses', 'lead', 'nitrates', 'nitrites', 'mercury',
    'perchlorate', 'radium', 'selenium', 'silver', 'uranium'
]

X = df[FEATURE_COLS].copy()
y = df['is_safe'].copy()

print(f"Feature matrix shape : {X.shape}")
print(f"Target vector shape  : {y.shape}")
print(f"Features in order    : {FEATURE_COLS}")

## Step 5 — Missing Value Imputation (features only)

In [ ]:
# Fit and transform on the ENTIRE feature dataset (no train/test split)
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)

print("Imputation complete.")
print(f"Any NaNs remaining: {np.isnan(X_imputed).any()}")

## Step 6 — Feature Scaling (StandardScaler / Z-score)

In [ ]:
# Fit on the full feature dataset and transform it
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print(f"Scaled matrix shape: {X_scaled.shape}")
print(f"Mean  (should be ~0): {X_scaled.mean(axis=0).round(4)}")
print(f"Std   (should be ~1): {X_scaled.std(axis=0).round(4)}")

## Step 7 & 8 — Model Specification and Training

In [ ]:
# Train on the FULL dataset — no train/test split, no shuffle
model = LogisticRegression(
    solver='liblinear',
    penalty='l2',
    C=1,
    random_state=42,
    max_iter=1000,
    tol=1e-8
)
model.fit(X_scaled, y)

print("Model trained successfully.")
print(f"Training accuracy: {model.score(X_scaled, y):.4f}")

## Step 9 — Decision Boundary Equation

In [ ]:
intercept = model.intercept_[0]
coefficients = model.coef_[0]

print("=" * 70)
print("DECISION BOUNDARY EQUATION")
print("=" * 70)
print(f"\nz = {intercept:.6f}")
for feat, coef in zip(FEATURE_COLS, coefficients):
    sign = '+' if coef >= 0 else '-'
    print(f"    {sign} {abs(coef):.6f} * {feat}")

# Compact single-line form
terms = [f"({coef:+.6f})*{feat}" for feat, coef in zip(FEATURE_COLS, coefficients)]
equation = f"z = {intercept:.6f} " + " ".join(terms)
print(f"\nCompact form:\n{equation}")

print("\n" + "=" * 70)
print("CLASSIFICATION RULE")
print("=" * 70)
print("  If z >= 0  →  Safe   (is_safe = 1)")
print("  If z <  0  →  Not Safe (is_safe = 0)")

## Step 10 — Coefficient Summary Table

In [ ]:
import pandas as pd

coef_df = pd.DataFrame({
    'feature'    : FEATURE_COLS,
    'coefficient': coefficients
}).sort_values('coefficient', ascending=False)

print(f"Intercept (β0): {intercept:.6f}\n")
print(coef_df.to_string(index=False))

## Visualise Coefficient Magnitudes

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
colors = ['steelblue' if c >= 0 else 'tomato' for c in coef_df['coefficient']]
plt.barh(coef_df['feature'], coef_df['coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Coefficient value')
plt.title('Logistic Regression Coefficients\n(blue = positive effect on safety, red = negative)')
plt.tight_layout()
plt.show()

print("\nBlue bars: feature increases probability of water being SAFE")
print("Red  bars: feature increases probability of water being UNSAFE")